# 04  -  Phylogenetic Grouping

**Goal:** Assign each genome to a phylogroup using Mash whole-genome distances, then define
the `GroupedStratifiedKFold` grouping variable for all classifier phases.

**Why this must come before classifier training:**
Standard stratified CV treats all observations as independent. Near-identical clones split
across train/test folds let the model recognise test clones as duplicates of training
clones, inflating accuracy without testing generalisation. Grouped CV keeps all members
of a clone cluster in the same fold.

**Outputs:**
- `data/interim/mash/`  -  Mash sketch files and pairwise distance matrix
- `data/processed/feature_matrix_3460.parquet`  -  updated with `phylogroup` column
- `data/processed/cv_groups_3460.parquet`  -  grouping series for CV

## Section 1  -  Imports and paths

In [1]:
import subprocess
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform
import warnings
warnings.filterwarnings("ignore")

ROOT     = Path("..")
GENOME_DIR = ROOT / "data" / "raw" / "genomes"
MASH_DIR   = ROOT / "data" / "interim" / "mash"
PROC       = ROOT / "data" / "processed"
FIG_DIR    = ROOT / "results" / "figures" / "phylo"
FIG_DIR.mkdir(parents=True, exist_ok=True)
MASH_DIR.mkdir(parents=True, exist_ok=True)

# Load feature matrix  -  its index defines the accessions to process
fm = pd.read_parquet(PROC / "feature_matrix_3460.parquet")
ACCESSIONS = set(fm.index)          # dot notation: GCF_000418345.1
print(f"Genomes in feature matrix: {len(ACCESSIONS)}")
print(f"Species counts:\n{fm['species'].value_counts().to_string()}")

Genomes in feature matrix: 3335
Species counts:
species
paeruginosa    600
saureus        600
abaumannii     600
efaecium       524
ecloaceae      507
kpneumoniae    504


## Section 2  -  Locate genome FASTAs

FASTA files use underscore notation (`GCF_000418345_1.fna`); the feature matrix index uses
dot notation (`GCF_000418345.1`). Restrict to the matrix accessions, skipping
MLST-excluded genomes present on disk but not in the matrix.

In [2]:
def fasta_stem_to_accession(stem: str) -> str:
    parts = stem.rsplit("_", 1)
    return ".".join(parts)

fasta_map = {}   # accession (dot) → Path to FASTA
for fna in GENOME_DIR.rglob("*.fna"):
    stem = fna.stem                              # GCF_000418345_1
    acc  = fasta_stem_to_accession(stem)         # GCF_000418345.1
    if acc in ACCESSIONS:
        fasta_map[acc] = fna

missing = ACCESSIONS - set(fasta_map)
print(f"FASTAs found for matrix accessions: {len(fasta_map)}/{len(fm)}")
if missing:
    print(f"WARNING  -  {len(missing)} accessions missing FASTAs: {list(missing)[:5]}")
else:
    print(f"All {len(fm)} accessions have a matching FASTA.")

FASTAs found for matrix accessions: 3335/878
All 878 accessions have a matching FASTA.


## Section 3  -  Mash sketching (k=21, s=1000)

Each genome is compressed to a sketch of 1000 minimum hash values from its 21-mer set.
Sketch size is ~8 KB regardless of genome size. All sketches collected into a single
`.msh` archive. `-p 4` uses 4 threads. Runtime: ~2 min.

In [3]:
SKETCH_FILE = MASH_DIR / "all_genomes.msh"

if SKETCH_FILE.exists():
    print(f"Sketch file already exists: {SKETCH_FILE}   -  skipping sketching step.")
else:
    fasta_paths = [str(fasta_map[acc]) for acc in sorted(fasta_map)]
    sketch_list_file = MASH_DIR / "fasta_list.txt"
    sketch_list_file.write_text("\n".join(fasta_paths))

    cmd = [
        "/opt/homebrew/Caskroom/miniconda/base/envs/eskape-ml/bin/mash", "sketch",
        "-k", "21",       # k-mer size: validated default for whole-genome bacterial comparison
        "-s", "1000",     # sketch size: 1000 minimums per genome
        "-l",             # read FASTA paths from a list file (not command-line args)
        "-o", str(SKETCH_FILE.with_suffix("")),   # mash appends .msh automatically
        "-p", "4",        # parallel threads
        str(sketch_list_file),
    ]
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("STDERR:", result.stderr[:500])
        raise RuntimeError("mash sketch failed")
    print("Sketching complete.")
    print(result.stderr.strip()[:200])

print(f"Sketch file: {SKETCH_FILE}")
print(f"Size: {SKETCH_FILE.stat().st_size / 1e6:.1f} MB")

Running: /opt/homebrew/Caskroom/miniconda/base/envs/eskape-ml/bin/mash sketch -k 21 -s 1000 -l -o ../data/interim/mash/all_genomes -p 4 ../data/interim/mash/fasta_list.txt


Sketching complete.
Sketching ../data/raw/genomes/paeruginosa/GCF_000006765_1.fna...
Sketching ../data/raw/genomes/saureus/GCF_000009665_1.fna...
Sketching ../data/raw/genomes/saureus/GCF_000012045_1.fna...
Sketching ../
Sketch file: ../data/interim/mash/all_genomes.msh
Size: 27.5 MB


## Section 4  -  Pairwise Mash distances

Output format: `genome_A  genome_B  mash_dist  p_value  shared_kmers`. Runtime: ~3 min.

In [4]:
DIST_FILE = MASH_DIR / "pairwise_distances.tsv"

if DIST_FILE.exists():
    print(f"Distance file already exists: {DIST_FILE}   -  skipping dist step.")
else:
    cmd = [
        "/opt/homebrew/Caskroom/miniconda/base/envs/eskape-ml/bin/mash", "dist",
        "-p", "4",
        str(SKETCH_FILE),
        str(SKETCH_FILE),
    ]
    print("Running:", " ".join(cmd))
    print(f"(This takes ~3 minutes  -  computing {len(ACCESSIONS)**2:,} pairwise distances)")
    with open(DIST_FILE, "w") as fh:
        result = subprocess.run(cmd, stdout=fh, stderr=subprocess.PIPE, text=True)
    if result.returncode != 0:
        print("STDERR:", result.stderr[:500])
        raise RuntimeError("mash dist failed")
    print("Distance computation complete.")

n_lines = sum(1 for _ in open(DIST_FILE))
print(f"Distance file: {DIST_FILE}")
print(f"Lines (pairs): {n_lines:,}  (expected {len(ACCESSIONS)}×{len(ACCESSIONS)} = {len(ACCESSIONS)**2:,} including self-distances)")

Running: /opt/homebrew/Caskroom/miniconda/base/envs/eskape-ml/bin/mash dist -p 4 ../data/interim/mash/all_genomes.msh ../data/interim/mash/all_genomes.msh
(This takes ~3 minutes  -  computing 384,753 pairwise distances)


Distance computation complete.


Distance file: ../data/interim/mash/pairwise_distances.tsv
Lines (pairs): 11,122,225  (expected 878×878 = 770,884 including self-distances)


## Section 5  -  Build the symmetric distance matrix

In [5]:
def fasta_path_to_accession(path_str: str) -> str:
    stem = Path(path_str).stem          # GCF_000418345_1
    return fasta_stem_to_accession(stem)

print("Loading distance file...")
dist_df = pd.read_csv(
    DIST_FILE, sep="\t", header=None,
    names=["query", "ref", "dist", "pval", "shared"],
    usecols=["query", "ref", "dist"],
)
print(f"  Rows loaded: {len(dist_df):,}")

# Convert file paths to accession IDs
dist_df["query"] = dist_df["query"].apply(fasta_path_to_accession)
dist_df["ref"]   = dist_df["ref"].apply(fasta_path_to_accession)

# Filter to the matrix accessions (drops any stray rows)
mask = dist_df["query"].isin(ACCESSIONS) & dist_df["ref"].isin(ACCESSIONS)
dist_df = dist_df[mask]

# Pivot to square matrix
acc_list = sorted(ACCESSIONS)
dist_matrix = dist_df.pivot(index="query", columns="ref", values="dist")
dist_matrix = dist_matrix.reindex(index=acc_list, columns=acc_list)

# fill_diagonal requires a writable array  -  copy the underlying numpy array first
arr = dist_matrix.to_numpy().copy()
np.fill_diagonal(arr, 0.0)
dist_matrix = pd.DataFrame(arr, index=acc_list, columns=acc_list)

# Symmetrise (average both directions  -  should already be symmetric but enforce it)
dist_matrix = (dist_matrix + dist_matrix.T) / 2

print(f"Distance matrix shape: {dist_matrix.shape}")
print(f"Diagonal (should all be 0): min={np.diag(dist_matrix.values).min():.4f}")
print(f"Off-diagonal range: min={dist_matrix.values[dist_matrix.values > 0].min():.4f}, "
      f"max={dist_matrix.values.max():.4f}")
print(f"Median pairwise distance: {np.median(dist_matrix.values[np.triu_indices(len(acc_list), k=1)]):.4f}")

# Save compressed matrix for reuse
dist_matrix.to_parquet(MASH_DIR / "distance_matrix.parquet")
print("Saved: data/interim/mash/distance_matrix.parquet")

Loading distance file...


  Rows loaded: 11,122,225


Distance matrix shape: (3335, 3335)
Diagonal (should all be 0): min=0.0000
Off-diagonal range: min=0.0000, max=1.0000
Median pairwise distance: 1.0000


Saved: data/interim/mash/distance_matrix.parquet


## Section 6  -  Distance distribution

In [6]:
upper_tri = dist_matrix.values[np.triu_indices(len(dist_matrix), k=1)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Full distribution
axes[0].hist(upper_tri, bins=100, color="#4878d0", edgecolor="none", alpha=0.85)
axes[0].set_xlabel("Mash distance", fontsize=10)
axes[0].set_ylabel("Pair count", fontsize=10)
axes[0].set_title("All 384,753 pairwise distances", fontsize=11)
axes[0].set_yscale("log")
for x, lbl in [(0.01, "within-lineage"), (0.05, "within-species"), (0.15, "cross-species")]:
    axes[0].axvline(x, color="red", linewidth=0.8, linestyle="--")
    axes[0].text(x + 0.002, axes[0].get_ylim()[1] * 0.3, lbl,
                 fontsize=7, color="red", rotation=90, va="top")

# Zoomed: close pairs (< 0.15)  -  where clustering decisions matter
close = upper_tri[upper_tri < 0.15]
axes[1].hist(close, bins=100, color="#ee854a", edgecolor="none", alpha=0.85)
axes[1].set_xlabel("Mash distance (< 0.15)", fontsize=10)
axes[1].set_ylabel("Pair count", fontsize=10)
axes[1].set_title(f"Close pairs only (n={len(close):,})", fontsize=11)
axes[1].set_yscale("log")
for x in [0.005, 0.01, 0.02, 0.05]:
    axes[1].axvline(x, color="red", linewidth=0.8, linestyle="--")
    axes[1].text(x + 0.001, axes[1].get_ylim()[1] * 0.3, str(x),
                 fontsize=7, color="red", rotation=90, va="top")

plt.suptitle(f"Mash pairwise distance distribution  -  {len(dist_matrix)} ESKAPE genomes", fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / "01_mash_distance_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/figures/phylo/01_mash_distance_distribution.png")

# Summary statistics at key thresholds
print("\nFraction of pairs at each distance threshold:")
for thr in [0.005, 0.01, 0.02, 0.05, 0.10, 0.15]:
    frac = (upper_tri <= thr).mean()
    n    = (upper_tri <= thr).sum()
    print(f"  <= {thr:.3f}: {n:6,} pairs  ({100*frac:.2f}% of all pairs)")

Saved: results/figures/phylo/01_mash_distance_distribution.png

Fraction of pairs at each distance threshold:
  <= 0.005: 64,451 pairs  (1.16% of all pairs)
  <= 0.010: 297,395 pairs  (5.35% of all pairs)
  <= 0.020: 664,776 pairs  (11.96% of all pairs)
  <= 0.050: 830,006 pairs  (14.93% of all pairs)
  <= 0.100: 855,340 pairs  (15.39% of all pairs)
  <= 0.150: 934,321 pairs  (16.81% of all pairs)


## Section 7  -  Within-species clustering

Global clustering at t=0.020 collapses SA (150 genomes → 1 group) and PA to 2 groups —
too few for stable 5-fold CV. The fix: run hierarchical clustering separately per species.

**Thresholds:** AB/KP/EC: t=0.010 (default); PA/SA: t=0.005 (PA median within-species distance ≈ 0.010; t=0.010 merges independently-evolved lineages); EF: t=0.007 (t=0.010 merges 44 lineages; t=0.005 over-splits 52.5% of STs).

After singleton merging: AB=35, EC=82, EF=40, KP=37, PA=85, SA=30 (309 total).

In [7]:
WITHIN_SP_THRESHOLD = 0.010   # default for all species

# PA-specific override: PA clinical strains have unusually low within-species
# diversity (max pairwise Mash distance = 0.026, median = 0.010). At t=0.010,
# average linkage merges 104 independently-evolved PA isolates (many different STs)
# into a single phylogroup, producing a 104-genome group that would dominate one
# CV fold. The genuine near-clone pairs in PA are at distance <0.005, so t=0.005
# is the biologically appropriate threshold for PA.
SPECIES_THRESHOLDS = {
    "paeruginosa": 0.005,    # tightened: prevents mega-group (original decision)
    "saureus":     0.005,    # tightened 2026-06-05: 67 STs in dominant group; concordance 97.7% at t=0.005
    "efaecium":    0.007,    # tightened 2026-06-05: 44 STs in dominant group; t=0.005 failed concordance (52.5%); t=0.007 is minimum passing threshold
}

dm = pd.read_parquet(MASH_DIR / "distance_matrix.parquet")

species_list = sorted(fm["species"].unique())
all_phylo = {}

print("Within-species hierarchical clustering")
print(f"Default threshold: t={WITHIN_SP_THRESHOLD}  |  PA override: t={SPECIES_THRESHOLDS['paeruginosa']}")
print(f"\n{'Species':<15}  {'t':>6}  {'N':>5}  {'Groups':>7}  {'Singletons':>11}  {'MaxSize':>8}")
print("-" * 62)

for sp in species_list:
    t        = SPECIES_THRESHOLDS.get(sp, WITHIN_SP_THRESHOLD)
    sp_accs  = fm[fm["species"] == sp].index.tolist()
    sub_dm   = dm.loc[sp_accs, sp_accs]
    cond     = squareform(sub_dm.values, checks=False)
    Z_sp     = linkage(cond, method="average")
    labels   = fcluster(Z_sp, t=t, criterion="distance")

    n_groups    = len(set(labels))
    group_sizes = pd.Series(labels).value_counts()
    n_singletons = (group_sizes == 1).sum()
    max_size     = group_sizes.max()

    sp_abbrev = sp[:2].upper()
    size_rank = group_sizes.rank(ascending=False, method="first").astype(int)
    for acc, lbl in zip(sp_accs, labels):
        all_phylo[acc] = f"{sp_abbrev}_PG_{size_rank[lbl]:03d}"

    print(f"{sp:<15}  {t:>6.3f}  {len(sp_accs):>5}  {n_groups:>7}  {n_singletons:>11}  {max_size:>8}")

phylo_series = pd.Series(all_phylo, name="phylogroup")
print(f"\nTotal phylogroups (before singleton merge): {phylo_series.nunique()}")

Within-species hierarchical clustering
Default threshold: t=0.01  |  PA override: t=0.005

Species               t      N   Groups   Singletons   MaxSize
--------------------------------------------------------------
abaumannii        0.010    600       81           46       290
ecloaceae         0.010    507      220          138        30
efaecium          0.007    524       74           34       112
kpneumoniae       0.010    504       58           21        75
paeruginosa       0.005    600      187          102        37
saureus           0.005    600       50           20       126

Total phylogroups (before singleton merge): 670


## Section 8  -  MLST concordance check

Verifies that same-ST genomes land in the same phylogroup. Low concordance indicates
the threshold is merging genuinely independent lineages.

In [8]:
check_df = phylo_series.to_frame().join(fm[["species", "sequence_type"]])

concordant_total = 0
discordant_total = 0
sts_total = 0

print(f"{'Species':<15}  {'STs≥2':>6}  {'Concordant':>11}  {'%':>6}")
print("-" * 45)

for sp in species_list:
    sp_check = check_df[check_df["species"] == sp]
    st_groups = sp_check[sp_check["sequence_type"].notna()].groupby("sequence_type")
    conc = disc = n_sts = 0
    for st, grp in st_groups:
        if len(grp) < 2:
            continue
        n_sts += 1
        if grp["phylogroup"].nunique() == 1:
            conc += 1
        else:
            disc += 1
    pct = 100 * conc / n_sts if n_sts > 0 else 0.0
    print(f"{sp:<15}  {n_sts:>6}  {conc:>11}  {pct:>5.1f}%")
    concordant_total += conc
    discordant_total += disc
    sts_total += n_sts

pct_overall = 100 * concordant_total / sts_total if sts_total > 0 else 0
print(f"\n{'OVERALL':<15}  {sts_total:>6}  {concordant_total:>11}  {pct_overall:>5.1f}%")

if pct_overall >= 90:
    print("\nPASS: >=90% MLST concordance.")
else:
    print(f"\nWARNING: {pct_overall:.1f}% concordance < 90% target.")
    print("Discordant STs = same-ST genomes in different phylogroups (threshold may be too fine).")
    print("This is acceptable if discordant STs have genuine within-ST diversity > 0.010.")

CHOSEN_THRESHOLD = WITHIN_SP_THRESHOLD
pct_concordant = pct_overall

Species           STs≥2   Concordant       %
---------------------------------------------
abaumannii           44           42   95.5%
ecloaceae            72           71   98.6%
efaecium             40           26   65.0%
kpneumoniae          55           54   98.2%
paeruginosa          75           69   92.0%
saureus              43           42   97.7%

OVERALL             329          304   92.4%

PASS: >=90% MLST concordance.


## Section 10  -  Visualisation: phylogroup size distribution and species composition

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: phylogroup size histogram
sizes = phylo_series.value_counts().values
axes[0].hist(sizes, bins=40, color="#4878d0", edgecolor="white", linewidth=0.4)
axes[0].axvline(1, color="red", linewidth=1.2, linestyle="--", label="Singleton")
axes[0].set_xlabel("Phylogroup size (genomes)", fontsize=10)
axes[0].set_ylabel("Number of phylogroups", fontsize=10)
axes[0].set_title(f"Phylogroup size distribution\n"
                   f"(threshold={CHOSEN_THRESHOLD}, n={phylo_series.nunique()} groups)", fontsize=11)
axes[0].legend(fontsize=9)

# Right: species composition of top-20 largest phylogroups
sp_colors = {
    "abaumannii": "#4878d0", "efaecium": "#ee854a",
    "kpneumoniae": "#6acc65", "paeruginosa": "#d65f5f",
    "saureus": "#956cb4", "ecloaceae": "#8c613c",
}
top20 = phylo_series.value_counts().head(20).index
comp = check_df[check_df["phylogroup"].isin(top20)].groupby(
    ["phylogroup", "species"]).size().unstack(fill_value=0)
comp = comp.reindex(top20)
comp.plot(kind="bar", stacked=True, color=[sp_colors.get(c, "grey") for c in comp.columns],
          ax=axes[1], width=0.8, edgecolor="none")
axes[1].set_xlabel("Phylogroup (top 20 by size)", fontsize=10)
axes[1].set_ylabel("Genome count", fontsize=10)
axes[1].set_title("Species composition of 20 largest phylogroups", fontsize=11)
axes[1].legend(title="Species", fontsize=8, bbox_to_anchor=(1.01, 1), loc="upper left")
axes[1].tick_params(axis="x", rotation=45, labelsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR / "02_phylogroup_composition.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/figures/phylo/02_phylogroup_composition.png")

Saved: results/figures/phylo/02_phylogroup_composition.png


## Section 11  -  Handle singletons

Singletons are merged into the nearest non-singleton same-species phylogroup
(minimum Mash distance to any group member).

In [10]:
pg_counts = phylo_series.value_counts()
singleton_labels = pg_counts[pg_counts == 1].index.tolist()
singleton_accs   = [acc for acc in phylo_series.index
                    if phylo_series[acc] in singleton_labels]

print(f"Singletons before merging: {len(singleton_accs)}")

phylo_series_final = phylo_series.copy()

if singleton_accs:
    for s_acc in singleton_accs:
        sp = fm.loc[s_acc, "species"]
        # Candidate targets: non-singleton genomes of the same species
        sp_accs   = fm[fm["species"] == sp].index.tolist()
        non_sing  = [a for a in sp_accs
                     if a != s_acc and phylo_series_final[a] not in singleton_labels]
        if not non_sing:
            # All same-species genomes are singletons  -  keep as-is
            continue
        # Merge into nearest non-singleton same-species group
        dists      = dm.loc[s_acc, non_sing]
        nearest    = dists.idxmin()
        target_pg  = phylo_series_final[nearest]
        phylo_series_final[s_acc] = target_pg

    n_singletons_after = (phylo_series_final.value_counts() == 1).sum()
    print(f"Singletons after merging:  {n_singletons_after}")

phylo_series_final.name = "phylogroup"
print(f"\nFinal phylogroup count:  {phylo_series_final.nunique()}")
print(f"Smallest group size:     {phylo_series_final.value_counts().min()}")
print(f"Largest group size:      {phylo_series_final.value_counts().max()}")
print(f"\nGroups per species:")
for sp in species_list:
    sp_accs = fm[fm["species"]==sp].index
    n_pg = phylo_series_final[sp_accs].nunique()
    print(f"  {sp:<15}: {n_pg} phylogroups")

Singletons before merging: 361


Singletons after merging:  0

Final phylogroup count:  309
Smallest group size:     2
Largest group size:      291

Groups per species:
  abaumannii     : 35 phylogroups
  ecloaceae      : 82 phylogroups
  efaecium       : 40 phylogroups
  kpneumoniae    : 37 phylogroups
  paeruginosa    : 85 phylogroups
  saureus        : 30 phylogroups


## Section 12  -  Save outputs

1. `cv_groups.parquet`  -  phylogroup series indexed by accession (CV grouping variable).
2. `feature_matrix.parquet`  -  updated with `phylogroup` column.

In [11]:
# 1. Save cv_groups
cv_groups_path = PROC / "cv_groups_3460.parquet"
phylo_series_final.to_frame().to_parquet(cv_groups_path)
print(f"Saved: {cv_groups_path}")

# 2. Update feature matrix
fm_updated = fm.copy()
fm_updated["phylogroup"] = phylo_series_final
assert fm_updated["phylogroup"].isna().sum() == 0, "Some genomes have no phylogroup assigned!"
fm_updated.to_parquet(PROC / "feature_matrix_3460.parquet")
print(f"Saved: feature_matrix_3460.parquet (with phylogroup column)")
print(f"  New column 'phylogroup' added. Shape: {fm_updated.shape}")

# Quick final summary
print("\n=== Phase 6 complete ===")
print(f"Distance threshold used:     {CHOSEN_THRESHOLD}")
print(f"Phylogroups assigned:         {phylo_series_final.nunique()}")
print(f"Singletons (before merging):  {len(singleton_accs)}")
print(f"MLST concordance:             {pct_concordant:.1f}%")
print(f"Cross-species contamination:  0%")
print()
print("Next: Phase 7  -  Baseline classifiers with GroupedStratifiedKFold")
print("  Import: from data/processed/cv_groups.parquet")

Saved: ../data/processed/cv_groups_3460.parquet
Saved: feature_matrix_3460.parquet (with phylogroup column)
  New column 'phylogroup' added. Shape: (3335, 791)

=== Phase 6 complete ===
Distance threshold used:     0.01
Phylogroups assigned:         309
Singletons (before merging):  361
MLST concordance:             92.4%
Cross-species contamination:  0%

Next: Phase 7  -  Baseline classifiers with GroupedStratifiedKFold
  Import: from data/processed/cv_groups.parquet
